# Notebook 04b: Vong 2 tinh chinh MLP — Tran_Hoang_Tung (E1)

Muc tim: tim mo hinh MLP tong quat tot nhat tren tap `val` (PR-AUC chinh, ROC-AUC phu, **khong chon theo Accuracy**).

Quy tat tu lop:
- Chi dung `train` va `val`. Tuyet doi khong mo `test` (ham doc se bao loi).
- Moi buoc hoc so lieu (chuan hoa, ma hoa...) nam trong `Pipeline`, chi fit tren `train`.
- Khong ghi de file cu: `04_models_Tung.ipynb`, `models/mlp.joblib`, `proba_val_mlp.npy`, `proba_val_mlp_balanced.npy`; tat ca file moi su dung hauto `_v2`.
- Chon mo hinh, tham so va ngưỡng chi tren `val`.


In [ ]:
import sys; sys.path.insert(0, "..")   # de import duoc thu muc src/

import json
import os
import time
from itertools import product
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from src.data import load_xy, SEED, REPO_ROOT
from src.features import build_preprocessor, check_columns
from src.evaluate import evaluate, evaluate_by_group, log_result, save_proba, load_proba

AUTHOR = "Tran_Hoang_Tung"
MODEL_NAME = "mlp_v2"


## A. Tai lap moc v1 (hidden=(64,), alpha=0.01)

Phai tra ve PR-AUC ~0.595 va ROC-AUC ~0.785 (sai so +-0.005). Neu lech hon, dung va bao.


In [ ]:
X_train, y_train = load_xy("train")
X_val, y_val = load_xy("val")
check_columns(X_train)   # bao loi neu co cot la hoac thieu cot
print(f"train {X_train.shape}, ty le huy {y_train.mean():.2%}")
print(f"val   {X_val.shape}, ty le huy {y_val.mean():.2%}")
print("so luoc CPU:", os.cpu_count())


In [ ]:
V1_PR, V1_ROC = 0.5950, 0.7845  # moc v1 da ghi tren notebook 04

def make_mlp(hidden, alpha, seed=SEED, lr=None, bs=None):
    kw = dict(hidden_layer_sizes=hidden, alpha=alpha, early_stopping=True,
              n_iter_no_change=10, max_iter=200, random_state=seed)
    if lr is not None:
        kw["learning_rate_init"] = lr
    if bs is not None:
        kw["batch_size"] = bs
    return Pipeline([("prep", build_preprocessor("linear")),
                     ("clf", MLPClassifier(**kw))])

def fit_score(hidden, alpha, seed=SEED, lr=None, bs=None):
    t0 = time.time()
    pipe = make_mlp(hidden, alpha, seed=seed, lr=lr, bs=bs).fit(X_train, y_train)
    dt = time.time() - t0
    proba = pipe.predict_proba(X_val)[:, 1]
    m = evaluate(y_val, proba)
    return pipe, proba, m, dt


In [ ]:
pipe_v1, proba_v1, m_v1, dt_v1 = fit_score((64,), 0.01, seed=SEED)
pr_v1 = m_v1["pr_auc"]; roc_v1 = m_v1["roc_auc"]
print(f"v1 lai lap: PR-AUC={pr_v1:.4f} (moc {V1_PR}), "
      f"ROC-AUC={roc_v1:.4f} (moc {V1_ROC}), thoi gian {dt_v1:.1f}s")
ok = abs(m_v1["pr_auc"] - V1_PR) <= 0.005 and abs(m_v1["roc_auc"] - V1_ROC) <= 0.005
print("PASS moc v1" if ok else "CANH BAO: v1 lai lap lech moc qua phep +-0.005")
if not ok:
    raise RuntimeError("Dung: lai lap v1 lech moc, khong tiep tuc Round 2.")


## B1. Mo rong luoi giai doan 1

hidden x alpha = 4 x 5 = 20 cau hinh, chon theo PR-AUC tren val.


In [ ]:
b1_rows = []
for hidden, alpha in product([(32,), (64,), (32, 16), (64, 32)],
                             [0.01, 0.03, 0.1, 0.3, 1.0]):
    _, _, m, dt = fit_score(hidden, alpha, seed=SEED)
    b1_rows.append({"hidden": str(hidden), "alpha": alpha,
                    "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "fit_s": round(dt, 1)})
b1 = pd.DataFrame(b1_rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
print(b1.round(4))
b1_fit_s = b1["fit_s"].sum()
b1_n = len(b1)
print(f"\nTong thoi gian B1: {b1_fit_s:.0f}s ({b1_fit_s/60:.1f} phut), {b1_n} cau hinh")
b1.to_csv(REPO_ROOT / "reports" / "results" / "_round2_b1.csv", index=False)


## B2. Giai doan 2: learning_rate_init & batch_size cho cau hinh tot nhat B1

Thu `learning_rate_init` x `batch_size`; chi mo rong them neu cau hinh tot nhat nam o **mep luoi** moi.


In [ ]:
def parse_hidden(s):
    """Robust: '(32,)' -> (32,), '(32, 16)' -> (32,16), '(64,)' -> (64,)."""
    s = str(s).strip("()")
    if s == "":
        return ()
    parts = [p for p in s.split(",") if p.strip() != ""]
    return tuple(int(p) for p in parts)

best_b1 = b1.iloc[0]
best_hidden = parse_hidden(best_b1["hidden"])
best_alpha = float(best_b1["alpha"])

b2_rows = []
for lr in [3e-4, 1e-3, 3e-3]:
    for bs in [64, 128, 256]:
        _, _, m, dt = fit_score(best_hidden, best_alpha, seed=SEED, lr=lr, bs=bs)
        b2_rows.append({"hidden": str(best_hidden), "alpha": best_alpha, "lr": lr, "bs": bs,
                        "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "fit_s": round(dt, 1)})
b2 = pd.DataFrame(b2_rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
print(b2.round(4))

best_b2 = b2.iloc[0]
at_edge = (best_b2["lr"] in [b2["lr"].min(), b2["lr"].max()] or
           best_b2["bs"] in [b2["bs"].min(), b2["bs"].max()])
print(f"\nTot nhat B2: {best_b2.to_dict()} | tai mep luoi: {at_edge}")

if at_edge:
    print("=> Mo rong them: cung cau hinh, them lr in [1e-4, 1e-2] va bs in [32, 512]")
    b2x_rows = []
    for lr in [1e-4, 1e-2]:
        for bs in [32, 512]:
            _, _, m, dt = fit_score(best_hidden, best_alpha, seed=SEED, lr=lr, bs=bs)
            b2x_rows.append({"hidden": str(best_hidden), "alpha": best_alpha, "lr": lr, "bs": bs,
                             "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "fit_s": round(dt, 1)})
    b2x = pd.DataFrame(b2x_rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
    print(b2x.round(4))
    b2 = pd.concat([b2, b2x], ignore_index=True).sort_values("pr_auc", ascending=False).reset_index(drop=True)
    best_b2 = b2.iloc[0]
    print(f"\nSau khi mo rong tot nhat: {best_b2.to_dict()}")

# Thong so tot nhat B1 (luon lay tu best_b1; B1 dung mac dinh MLPClassifier: lr=None, bs=None)
b1_hidden = parse_hidden(best_b1["hidden"])
b1_alpha = float(best_b1["alpha"])
b1_lr, b1_bs = None, None
b1_pr, b1_roc = best_b1["pr_auc"], best_b1["roc_auc"]

# Thong so tot nhat B2 (luon lay tu best_b2, khong phu thuoc vao viec chon B1 hay B2)
b2_hidden = parse_hidden(best_b2["hidden"])
b2_alpha = float(best_b2["alpha"])
b2_lr = best_b2["lr"] if "lr" in b2.columns else None
b2_bs = int(best_b2["bs"]) if "bs" in b2.columns else None
b2_pr, b2_roc = best_b2["pr_auc"], best_b2["roc_auc"]

# Chon cau hinh co PR-AUC cao nhat giua B1 va B2
use_b1 = bool(b1_pr >= b2_pr)
if use_b1:
    final_hidden, final_alpha = b1_hidden, b1_alpha
    final_lr, final_bs = b1_lr, b1_bs
    final_pr, final_roc = b1_pr, b1_roc
    final_stage = "B1"
else:
    final_hidden, final_alpha = b2_hidden, b2_alpha
    final_lr, final_bs = b2_lr, b2_bs
    final_pr, final_roc = b2_pr, b2_roc
    final_stage = "B2"
print(f"\nCau hinh chon: {final_stage} -> hidden={final_hidden}, alpha={final_alpha}, lr={final_lr}, bs={final_bs}")
print(f"PR-AUC chon = {final_pr:.4f} | ROC-AUC = {final_roc:.4f}")
print(f"Tong so cau hinh da chay (B1+B2): {len(b1) + len(b2)}")

# So sanh truc tiep tot nhat B1 vs tot nhat B2
print("\n=== So sanh tot nhat B1 vs tot nhat B2 ===")
print("B1 tot nhat: hidden=%s alpha=%.2f lr=%s bs=%s PR-AUC=%.4f ROC-AUC=%.4f" % (
    b1_hidden, b1_alpha, b1_lr, b1_bs, b1_pr, b1_roc))
print("B2 tot nhat: hidden=%s alpha=%s lr=%s bs=%s PR-AUC=%.4f ROC-AUC=%.4f" % (
    b2_hidden, b2_alpha, b2_lr, b2_bs, b2_pr, b2_roc))
print("B2 tot hon B1 (PR-AUC): %s (lech %.4f)" % (
    bool(b2_pr > b1_pr), b2_pr - b1_pr))

b2.to_csv(REPO_ROOT / "reports" / "results" / "_round2_b2.csv", index=False)



## C. Do nhieu: chay lai cau hinh chon voi >= 5 random_state khac nhau

Bao mean va std cua PR-AUC va ROC-AUC. Cau hinh duoc coi "tot hon v1" neu muc doi chuong > std.


In [ ]:
seeds = [42, 43, 44, 45, 46]
c_rows, models_c, probas_c = [], [], []
for s in seeds:
    pipe, proba, m, dt = fit_score(final_hidden, final_alpha, seed=s, lr=final_lr, bs=final_bs)
    c_rows.append({"seed": s, "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "fit_s": round(dt, 1)})
    models_c.append(pipe)
    probas_c.append(proba)
c = pd.DataFrame(c_rows)
print(c.round(4))
c_pr_mean = c["pr_auc"].mean(); c_pr_std = c["pr_auc"].std()
c_roc_mean = c["roc_auc"].mean(); c_roc_std = c["roc_auc"].std()
print(f"\nMean PR-AUC = {c_pr_mean:.4f} (+- {c_pr_std:.4f})")
print(f"Mean ROC-AUC = {c_roc_mean:.4f} (+- {c_roc_std:.4f})")
gain_pr = c["pr_auc"].mean() - V1_PR
print(f"Chenh chenh so vs v1: {gain_pr:+.4f} PR-AUC |")
print(f"'Tot hon v1' neu gain > std PR-AUC ({c_pr_std:.4f}): {bool(gain_pr > c_pr_std)}")
c.to_csv(REPO_ROOT / "reports" / "results" / "_round2_c.csv", index=False)


## D. Ensemble: trung binh xac suat cua 5-10 mo hinh (cung cau hinh, seed khac)


In [ ]:
proba_ens = np.mean(np.stack(probas_c), axis=0)
m_ens = evaluate(y_val, proba_ens)

# Mo hinh don voi seed chuan SEED = 42
pipe_single, proba_single, m_single, _ = fit_score(final_hidden, final_alpha,
                                                   seed=SEED, lr=final_lr, bs=final_bs)

cmp = pd.DataFrame([
    {"thu_tu": f"Don (seed {SEED})", "pr_auc": m_single["pr_auc"], "roc_auc": m_single["roc_auc"]},
    {"thu_tu": f"Ensemble (mean cua {len(seeds)} mo hinh)", "pr_auc": m_ens["pr_auc"], "roc_auc": m_ens["roc_auc"]},
])
print(cmp.round(4))
ens_better = bool(m_ens["pr_auc"] > m_single["pr_auc"])
print("Ensemble tot hon mo model don:", ens_better)


## E. Luu ket cuoi (khong ghi de v1)


In [ ]:
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# --- Mo hinh don tot nhat (seed chuan SEED = 42) ---
joblib.dump(pipe_single, MODELS_DIR / "mlp_v2.joblib")
save_proba("mlp_v2", proba_single, split="val")   # models/proba_val_mlp_v2.npy, thu tu giong val.csv

m05 = evaluate(y_val, proba_single, threshold=0.5)
log_result(AUTHOR, "E1", MODEL_NAME, "val", m05,
           note=f"round2: hidden={final_hidden}, alpha={final_alpha}, lr={final_lr}, bs={final_bs}, seed={SEED}")

print("=== Bang ket qua theo khach san (mo don mlp_v2, val) ===")
grp = evaluate_by_group(y_val, proba_single, X_val["hotel"]).round(3)
print(grp.to_string())

# --- Kiem tra file da luu: thu tu va khoi phuc ---
pr_loaded = load_proba(
    "mlp_v2", split="val")
m_reload = evaluate(y_val, pr_loaded)
print(f"\nKiem tra luu proba: shape={pr_loaded.shape},")
print(f"  khop thu tu val={len(pr_loaded) == len(X_val)},")
print(f"  giong nguyen goc={np.allclose(pr_loaded, proba_single)}")
pr_rl = m_reload["pr_auc"]; roc_rl = m_reload["roc_auc"]
print(f"Khoi phuc tu .npy: PR-AUC={pr_rl:.4f}, ROC-AUC={roc_rl:.4f}")


In [ ]:
if ens_better:
    save_proba("mlp_v2_ens", proba_ens, split="val")   # models/proba_val_mlp_v2_ens.npy
    meta = {
        "seeds": seeds,
        "config": {"hidden": list(final_hidden), "alpha": final_alpha,
                   "lr": final_lr, "bs": final_bs},
        "n_train": int(X_train.shape[0]), "n_val": int(X_val.shape[0]),
        "reload": "joblib.load('models/mlp_v2_ens_seed{s}.joblib') cho tung s trong seeds;",
        "reload2": "trung binh predict_proba(X_val)[:, 1] cua tat ca mo hinh.",
    }
    (MODELS_DIR / "mlp_v2_ensemble_meta.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    for s, pipe in zip(seeds, models_c):
        joblib.dump(pipe, MODELS_DIR / f"mlp_v2_ens_seed{s}.joblib")
    log_result(AUTHOR, "E1", "mlp_v2_ens", "val", m_ens,
               note=f"round2: ensemble mean of {len(seeds)} models, {final_hidden}, alpha={final_alpha}")
    print("Da luu ensemble: proba_val_mlp_v2_ens.npy, mlp_v2_ens_seed{42..46}.joblib, mlp_v2_ensemble_meta.json")
else:
    print("Ensemble khong tot hon mo model don -> khong luu file ensemble.")


## Tong ket: bang in san de dong bo voi Claude


In [ ]:
print("(a) Toan bo cau hinh da chay (B1 + B2):")
all_grid = pd.concat([b1.assign(stage="B1"), b2.assign(stage="B2")], ignore_index=True)
print(all_grid.round(4).to_string())

print("\n(b) Mean/std qua 5 random_state (cau hinh chon):")
print(c.round(4).to_string())
print(f"   Mean PR-AUC = {c_pr_mean:.4f} (+- {c_pr_std:.4f})")
print(f"   Mean ROC-AUC = {c_roc_mean:.4f} (+- {c_roc_std:.4f})")

print("\n(c) So sanh mo don vs ensemble (val):")
print(cmp.round(4).to_string())

print("\n(d) Ket qua theo khach san (mo don mlp_v2, val):")
print(grp.to_string())

print("\n(e) File da luu thu muc models/:")
for p in sorted(MODELS_DIR.glob("mlp_v2*")):
    print(" -", p.name, p.stat().st_size, "bytes")

print("\n(e) Test-Path: khoi phuc proba tu .npy va tinh lai chi so == ket qua da log:",
      np.allclose(pr_loaded, proba_single) and abs(m_reload["pr_auc"] - m05["pr_auc"]) < 1e-6)

print("\nDong log_result moi nhat (reports/results/Tran_Hoang_Tung.csv):")
res_csv = REPO_ROOT / "reports" / "results" / f"{AUTHOR}.csv"
res = pd.read_csv(res_csv)
print(res.tail(4).to_string())
